# 🎬 Netflix Dataset — Exploratory Data Analysis
**Dataset:** `cleaned_dataset.csv` — 8,807 Netflix titles (Movies & TV Shows)

**Columns:** `show_id`, `type`, `title`, `director`, `cast`, `country`, `date_added`, `release_year`, `rating`, `duration`, `listed_in`, `description`

---
### 📋 Table of Contents
1. Setup & Data Loading
2. Data Overview & Info
3. Missing Value Analysis
4. Univariate Analysis
5. Content Type Analysis (Movies vs TV Shows)
6. Year & Date Trends
7. Country Analysis
8. Genre / Category Analysis
9. Ratings Analysis
10. Duration Analysis
11. Director & Cast Insights
12. Word Cloud — Descriptions
13. Correlation & Summary

## 1. 📦 Setup & Data Loading

In [ ]:
# Install extra libraries (only needed in Colab)
!pip install wordcloud plotly --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans'
})
sns.set_palette('Set2')
print('✅ All libraries imported successfully!')

In [ ]:
# ── Upload your file in Colab using the Files panel (left sidebar)
# ── Or mount Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/cleaned_dataset.csv')

# ── Direct upload via Colab widget:
# from google.colab import files
# uploaded = files.upload()

# ── Load the dataset (update path if needed)
df = pd.read_csv('cleaned_dataset.csv')

print(f'✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 2. 🔍 Data Overview & Info

In [ ]:
print('=' * 55)
print('📐 SHAPE')
print(f'   Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')

print('\n📋 COLUMN DTYPES')
print(df.dtypes)

print('\n📊 BASIC INFO')
df.info()

In [ ]:
# ── Date Parsing
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month
df['month_name'] = df['date_added'].dt.strftime('%b')

# ── Duration parsing
df['duration_int'] = df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_unit'] = df['duration'].str.extract(r'(min|Season)')

# ── Replace 'nan' strings with actual NaN
df.replace('nan', np.nan, inplace=True)

print('✅ Preprocessing done!')
df[['date_added', 'year_added', 'month_added', 'duration_int', 'duration_unit']].head()

## 3. 🕳️ Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)
missing_df = missing_df[missing_df['Missing Count'] > 0]
print(missing_df)

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#e74c3c' if p > 30 else '#e67e22' if p > 10 else '#3498db'
          for p in missing_df['Missing %']]
bars = ax.barh(missing_df.index, missing_df['Missing %'], color=colors)
for bar, val in zip(bars, missing_df['Missing %']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=10)
ax.set_xlabel('Missing %')
ax.set_title('Missing Values per Column', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. 📊 Univariate Analysis

In [ ]:
print('🎬 Type Distribution:')
print(df['type'].value_counts())
print()
print('⭐ Rating Distribution:')
print(df['rating'].value_counts().head(10))
print()
print('📅 Release Year Range:')
print(f"   Min: {df['release_year'].min()}  |  Max: {df['release_year'].max()}")
print()
print('📅 Year Added Range:')
print(f"   Min: {df['year_added'].min()}  |  Max: {df['year_added'].max()}")

## 5. 🎬 Content Type Analysis — Movies vs TV Shows

In [ ]:
type_counts = df['type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie Chart
axes[0].pie(
    type_counts,
    labels=type_counts.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=['#E50914', '#221F1F'],
    textprops={'color': 'white', 'fontsize': 12},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Content Type Split', fontsize=13, fontweight='bold')

# Bar Chart
bars = axes[1].bar(type_counts.index, type_counts.values,
                   color=['#E50914', '#221F1F'], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, type_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('Count of Movies vs TV Shows', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')

plt.suptitle('Netflix Content Type Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. 📅 Year & Date Trends

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── Content added per year
year_added = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)
year_added.plot(kind='bar', ax=axes[0, 0], color=['#E50914', '#564d4d'],
                edgecolor='white', linewidth=0.5)
axes[0, 0].set_title('Titles Added to Netflix per Year', fontweight='bold')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].legend(title='Type')

# ── Release year distribution
release_filtered = df[df['release_year'] >= 1990]
axes[0, 1].hist(release_filtered['release_year'], bins=30,
                color='#E50914', edgecolor='white', linewidth=0.5)
axes[0, 1].set_title('Distribution of Release Years (1990+)', fontweight='bold')
axes[0, 1].set_xlabel('Release Year')
axes[0, 1].set_ylabel('Count')

# ── Month-wise addition
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_counts = df['month_name'].value_counts().reindex(month_order, fill_value=0)
axes[1, 0].bar(month_counts.index, month_counts.values,
               color='#3498db', edgecolor='white', linewidth=0.5)
axes[1, 0].set_title('Titles Added by Month', fontweight='bold')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Count')

# ── Gap between release year and year added
df['gap_years'] = df['year_added'] - df['release_year']
gap_filtered = df[df['gap_years'].between(0, 50)]
axes[1, 1].hist(gap_filtered['gap_years'], bins=30,
                color='#2ecc71', edgecolor='white', linewidth=0.5)
axes[1, 1].set_title('Gap: Release Year → Added to Netflix (years)', fontweight='bold')
axes[1, 1].set_xlabel('Years Gap')
axes[1, 1].set_ylabel('Count')

plt.suptitle('Netflix Year & Date Trends', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. 🌍 Country Analysis

In [ ]:
# Extract primary country (first listed)
df['primary_country'] = df['country'].str.split(',').str[0].str.strip()

top_countries = df['primary_country'].value_counts().dropna().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Top 15 countries
colors = ['#E50914'] + ['#564d4d'] * 14
bars = axes[0].barh(top_countries.index[::-1], top_countries.values[::-1], color=colors[::-1])
for bar, val in zip(bars, top_countries.values[::-1]):
    axes[0].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)
axes[0].set_title('Top 15 Countries by Content Count', fontweight='bold')
axes[0].set_xlabel('Number of Titles')

# ── Movies vs TV Shows by top 10 countries
top10 = top_countries.head(10).index.tolist()
country_type = df[df['primary_country'].isin(top10)].groupby(
    ['primary_country', 'type']).size().unstack(fill_value=0)
country_type.sort_values('Movie', ascending=True).plot(
    kind='barh', ax=axes[1], color=['#E50914', '#221F1F'],
    edgecolor='white', linewidth=0.5)
axes[1].set_title('Movies vs TV Shows — Top 10 Countries', fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].legend(title='Type')

plt.suptitle('Netflix Country Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Interactive Choropleth Map (Plotly)
country_counts = df['primary_country'].value_counts().reset_index()
country_counts.columns = ['country', 'count']

fig = px.choropleth(
    country_counts,
    locations='country',
    locationmode='country names',
    color='count',
    color_continuous_scale='Reds',
    title='🌍 Netflix Content Distribution by Country',
    hover_name='country',
    hover_data={'count': True}
)
fig.update_layout(height=450, title_font_size=15)
fig.show()

## 8. 🎭 Genre / Category Analysis

In [ ]:
# Explode genres (each title can have multiple)
genres = df['listed_in'].dropna().str.split(',').explode().str.strip()
top_genres = genres.value_counts().head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── All content
colors = plt.cm.Set3(np.linspace(0, 1, len(top_genres)))
axes[0].barh(top_genres.index[::-1], top_genres.values[::-1], color=colors)
axes[0].set_title('Top 20 Genres (All Content)', fontweight='bold')
axes[0].set_xlabel('Count')

# ── Movies only genres
movie_genres = df[df['type'] == 'Movie']['listed_in'].dropna().str.split(',').explode().str.strip()
top_movie_genres = movie_genres.value_counts().head(12)
axes[1].bar(range(len(top_movie_genres)), top_movie_genres.values,
            color='#E50914', edgecolor='white', linewidth=0.5)
axes[1].set_xticks(range(len(top_movie_genres)))
axes[1].set_xticklabels(top_movie_genres.index, rotation=45, ha='right')
axes[1].set_title('Top 12 Movie Genres', fontweight='bold')
axes[1].set_ylabel('Count')

plt.suptitle('Netflix Genre Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Interactive Treemap
genre_df = top_genres.head(25).reset_index()
genre_df.columns = ['genre', 'count']

fig = px.treemap(
    genre_df,
    path=['genre'],
    values='count',
    title='🎭 Genre Treemap — Top 25',
    color='count',
    color_continuous_scale='RdBu'
)
fig.update_layout(height=500, title_font_size=15)
fig.show()

## 9. ⭐ Ratings Analysis

In [ ]:
rating_order = ['G', 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'PG', 'TV-G', 'TV-PG',
                'PG-13', 'TV-14', 'R', 'TV-MA', 'NC-17', 'NR', 'UR']

rating_counts = df['rating'].value_counts().reindex(rating_order).dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Overall rating distribution
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(rating_counts)))
axes[0].bar(rating_counts.index, rating_counts.values, color=colors, edgecolor='white')
for i, (idx, val) in enumerate(zip(rating_counts.index, rating_counts.values)):
    axes[0].text(i, val + 20, str(val), ha='center', fontsize=8)
axes[0].set_title('Rating Distribution (All Content)', fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# ── Ratings split by type
rating_type = df.groupby(['rating', 'type']).size().unstack(fill_value=0)
rating_type = rating_type.reindex(rating_order).dropna()
rating_type.plot(kind='bar', ax=axes[1], color=['#E50914', '#564d4d'],
                 edgecolor='white', linewidth=0.5)
axes[1].set_title('Ratings by Content Type', fontweight='bold')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Type')

plt.suptitle('Netflix Ratings Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 10. ⏱️ Duration Analysis

In [ ]:
movies = df[df['type'] == 'Movie'].copy()
tv     = df[df['type'] == 'TV Show'].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Movie durations (minutes)
movie_dur = movies['duration_int'].dropna()
axes[0].hist(movie_dur, bins=40, color='#E50914', edgecolor='white', linewidth=0.5)
axes[0].axvline(movie_dur.mean(), color='black', linestyle='--',
                linewidth=1.5, label=f'Mean: {movie_dur.mean():.0f} min')
axes[0].axvline(movie_dur.median(), color='gold', linestyle='--',
                linewidth=1.5, label=f'Median: {movie_dur.median():.0f} min')
axes[0].set_title('Movie Duration Distribution (minutes)', fontweight='bold')
axes[0].set_xlabel('Duration (min)')
axes[0].set_ylabel('Count')
axes[0].legend()

# ── TV Show seasons
tv_seasons = tv['duration_int'].dropna().value_counts().sort_index().head(15)
axes[1].bar(tv_seasons.index.astype(int), tv_seasons.values,
            color='#3498db', edgecolor='white', linewidth=0.5)
for i, val in zip(tv_seasons.index, tv_seasons.values):
    axes[1].text(i, val + 5, str(val), ha='center', fontsize=9)
axes[1].set_title('TV Show — Number of Seasons', fontweight='bold')
axes[1].set_xlabel('Seasons')
axes[1].set_ylabel('Count')

plt.suptitle('Netflix Duration Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'\n🎬 Movie Duration Stats:')
print(movies['duration_int'].describe().round(1))
print(f'\n📺 TV Show Season Stats:')
print(tv['duration_int'].describe().round(1))

In [ ]:
# ── Box plot: movie durations by rating
top_ratings = df['rating'].value_counts().head(8).index
movie_rating_dur = movies[movies['rating'].isin(top_ratings)]

fig = px.box(
    movie_rating_dur,
    x='rating', y='duration_int',
    color='rating',
    title='🎬 Movie Duration Distribution by Rating',
    labels={'duration_int': 'Duration (min)', 'rating': 'Rating'}
)
fig.update_layout(height=450, showlegend=False, title_font_size=14)
fig.show()

## 11. 🎥 Director & Cast Insights

In [ ]:
# ── Top Directors
top_directors = df['director'].dropna().value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = plt.cm.Blues(np.linspace(0.4, 1.0, len(top_directors)))
axes[0].barh(top_directors.index[::-1], top_directors.values[::-1], color=colors)
for i, val in enumerate(top_directors.values[::-1]):
    axes[0].text(val + 0.1, i, str(val), va='center', fontsize=9)
axes[0].set_title('Top 15 Directors by Title Count', fontweight='bold')
axes[0].set_xlabel('Number of Titles')

# ── Top Cast Members
cast_all = df['cast'].dropna().str.split(',').explode().str.strip()
top_cast = cast_all.value_counts().head(15)

colors2 = plt.cm.Oranges(np.linspace(0.4, 1.0, len(top_cast)))
axes[1].barh(top_cast.index[::-1], top_cast.values[::-1], color=colors2)
for i, val in enumerate(top_cast.values[::-1]):
    axes[1].text(val + 0.1, i, str(val), va='center', fontsize=9)
axes[1].set_title('Top 15 Cast Members by Appearances', fontweight='bold')
axes[1].set_xlabel('Number of Titles')

plt.suptitle('Netflix Director & Cast Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 12. ☁️ Word Cloud — Descriptions & Titles

In [ ]:
stopwords_extra = {
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to',
    'for', 'of', 'with', 'by', 'from', 'is', 'are', 'was', 'were',
    'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did',
    'will', 'would', 'could', 'should', 'may', 'might', 'must', 'can',
    'his', 'her', 'their', 'its', 'he', 'she', 'they', 'who', 'when',
    'what', 'where', 'that', 'this', 'as', 'it', 'into', 'after',
    'while', 'which'
}

desc_text  = ' '.join(df['description'].dropna().tolist())
title_text = ' '.join(df['title'].dropna().tolist())

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, text, title, bg, cmap in zip(
    axes,
    [desc_text, title_text],
    ['Description Word Cloud', 'Title Word Cloud'],
    ['#0d0d0d', '#1a0000'],
    ['Reds', 'YlOrRd']
):
    wc = WordCloud(
        width=800, height=500,
        background_color=bg,
        stopwords=stopwords_extra,
        colormap=cmap,
        max_words=150,
        collocations=False
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontweight='bold', fontsize=13, pad=10)

plt.suptitle('Netflix Word Clouds', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 13. 📈 Trends Over Time (Interactive)

In [ ]:
# ── Growth of content added per year (interactive)
yearly_type = df.groupby(['year_added', 'type']).size().reset_index(name='count')
yearly_type = yearly_type.dropna(subset=['year_added'])

fig = px.line(
    yearly_type,
    x='year_added', y='count',
    color='type',
    markers=True,
    title='📈 Netflix Content Added Over the Years',
    labels={'year_added': 'Year', 'count': 'Titles Added', 'type': 'Type'},
    color_discrete_map={'Movie': '#E50914', 'TV Show': '#3498db'}
)
fig.update_layout(height=430, title_font_size=14)
fig.show()

In [ ]:
# ── Sunburst: Type → Rating → Country
sunburst_df = df[['type', 'rating', 'primary_country']].dropna()
sunburst_df = sunburst_df[sunburst_df['primary_country'].isin(
    df['primary_country'].value_counts().head(10).index)]

fig = px.sunburst(
    sunburst_df,
    path=['type', 'rating', 'primary_country'],
    title='☀️ Sunburst: Content Type → Rating → Country (Top 10)',
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig.update_layout(height=550, title_font_size=14)
fig.show()

## 14. 📋 Summary & Key Insights

In [ ]:
print('=' * 60)
print('         📊  NETFLIX DATASET — KEY INSIGHTS')
print('=' * 60)
print(f'\n📦 Total Titles       : {len(df):,}')
print(f'🎬 Movies             : {(df["type"]=="Movie").sum():,} ({(df["type"]=="Movie").mean()*100:.1f}%)')
print(f'📺 TV Shows           : {(df["type"]=="TV Show").sum():,} ({(df["type"]=="TV Show").mean()*100:.1f}%)')
print(f'\n📅 Earliest Release   : {df["release_year"].min()}')
print(f'📅 Latest Release     : {df["release_year"].max()}')
print(f'\n🌍 Top Country        : {df["primary_country"].value_counts().index[0]} ({df["primary_country"].value_counts().iloc[0]:,} titles)')
print(f'⭐ Top Rating         : {df["rating"].value_counts().index[0]} ({df["rating"].value_counts().iloc[0]:,} titles)')
print(f'🎭 Top Genre          : {genres.value_counts().index[0]} ({genres.value_counts().iloc[0]:,} titles)')
print(f'🎥 Most Active Dir.   : {df["director"].value_counts().index[0]} ({df["director"].value_counts().iloc[0]} titles)')
avg_movie = movies['duration_int'].mean()
print(f'⏱️  Avg Movie Duration : {avg_movie:.0f} minutes ({avg_movie/60:.1f} hrs)')
print(f'📺 1-Season TV Shows  : {(tv["duration_int"]==1).sum():,} ({(tv["duration_int"]==1).mean()*100:.1f}%)')
print(f'\n🕳️  Missing — Director : {df["director"].isna().mean()*100:.1f}%')
print(f'🕳️  Missing — Cast     : {df["cast"].isna().mean()*100:.1f}%')
print(f'🕳️  Missing — Country  : {df["country"].isna().mean()*100:.1f}%')
print('=' * 60)